# Vanilla BiRNN for Protein Structure Prediction

This notebook implements a Vanilla Bidirectional RNN (using `nn.RNN`) for Sequence-to-Sequence (Seq2Seq) prediction of protein secondary structures (Q8 and Q3).

## Task Overview
- **Input**: Amino acid sequence.
- **Output 1**: Q8 Secondary Structure (8 classes).
- **Output 2**: Q3 Secondary Structure (3 classes).
- **Metric**: Harmonic Mean of Token-Level F1 scores (excluding Padding).

In [1]:
# !pip install trackio


import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# --- Configuration ---
class Config:
    seed = 42
    device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
    batch_size = 64
    embed_dim = 128
    hidden_dim = 256
    num_layers = 2
    learning_rate = 0.001
    epochs = 10
    
    # Paths
    train_path = "sep-25-dl-gen-ai-nppe-2/train.csv"
    test_path = "sep-25-dl-gen-ai-nppe-2/test.csv"
    
    # TrackIO
    trackio_project = "25-t3-nppe2"
    trackio_space_id = "manyakaistha/dlgenai-nppe"
    trackio_group = "vanilla_birnn"  # Optional grouping

cfg = Config()
print(f"Using device: {cfg.device}")

# Set seeds
def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)


Using device: mps


/Volumes/1700 APFS/PERSONAL_PROGRAMMING_PROJECTS/intro_dl_nppe/nppe2/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# --- TrackIO Setup ---
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "hf_placeholder_token_for_reproducibility"
os.environ["TRACKIO_SPACE_ID"] = cfg.trackio_space_id

In [ ]:
def maybe_init_trackio():
    try:
        import trackio
        token = os.environ.get("HUGGINGFACEHUB_API_TOKEN") or os.environ.get("HF_TOKEN")
        if cfg.trackio_space_id and token:
            trackio.init(
                project=cfg.trackio_project,
                space_id=cfg.trackio_space_id,
                group=cfg.trackio_group
            )
            trackio.log({"status/started": 1})
            print("✓ TrackIO initialized")
            return trackio
    except Exception as e:
        print(f"TrackIO initialization failed: {e}")
        try:
             # Fallback if trackio not installed, try installing it or just pass
             # print("Attempting to install trackio...")
             !pip install trackio  # Uncomment if needed
             pass
        except:
             pass
    return None

trk = maybe_init_trackio()

* Trackio project initialized: 25-t3-nppe2
* Trackio metrics will be synced to Hugging Face Dataset: manyakaistha/dlgenai-nppe-dataset
* Found existing space: https://huggingface.co/spaces/manyakaistha/dlgenai-nppe
* View dashboard by going to: https://manyakaistha-dlgenai-nppe.hf.space/


* Created new run: manyakaistha-1765617483
✓ TrackIO initialized


## Data Preprocessing

In [4]:
# --- Mappings ---
# Based on unique characters found in dataset:
# Q8 chars: ['B', 'C', 'E', 'G', 'H', 'I', 'S', 'T']
# Q3 chars: ['C', 'E', 'H']
# Seq chars: ['*', 'A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']

PAD_TOKEN = 0

# 1. Amino Acid Map
AA_CHARS = ['<PAD>', '<UNK>', '*', 'A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']
AA_TO_ID = {c: i for i, c in enumerate(AA_CHARS)}
ID_TO_AA = {i: c for c, i in AA_TO_ID.items()}

# 2. Q8 Map
Q8_CHARS = ['<PAD>'] + sorted(['B', 'C', 'E', 'G', 'H', 'I', 'S', 'T'])
Q8_TO_ID = {c: i for i, c in enumerate(Q8_CHARS)}
ID_TO_Q8 = {i: c for c, i in Q8_TO_ID.items()}

# 3. Q3 Map
Q3_CHARS = ['<PAD>'] + sorted(['C', 'E', 'H'])
Q3_TO_ID = {c: i for i, c in enumerate(Q3_CHARS)}
ID_TO_Q3 = {i: c for c, i in Q3_TO_ID.items()}

print(f"Vocab Sizes: AA={len(AA_TO_ID)}, Q8={len(Q8_TO_ID)}, Q3={len(Q3_TO_ID)}")

# --- Dataset ---
class ProteinDataset(Dataset):
    def __init__(self, csv_path, mode='train'):
        self.data = pd.read_csv(csv_path)
        self.mode = mode
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        seq_str = row['seq']
        
        # Encode sequence
        seq_ids = [AA_TO_ID.get(c, AA_TO_ID['<UNK>']) for c in seq_str]
        
        item = {
            'seq': torch.tensor(seq_ids, dtype=torch.long),
            'id': row['id']
        }
        
        if self.mode == 'train':
            q8_str = row['sst8']
            q3_str = row['sst3']
            
            q8_ids = [Q8_TO_ID.get(c, 0) for c in q8_str] # Fallback 0 is PAD?? Should be error or handle carefully. Assuming clean data.
            q3_ids = [Q3_TO_ID.get(c, 0) for c in q3_str]
            
            item['q8'] = torch.tensor(q8_ids, dtype=torch.long)
            item['q3'] = torch.tensor(q3_ids, dtype=torch.long)
            
        return item

# --- Collate Function ---
def collate_fn(batch):
    # Pad sequences
    seqs = [item['seq'] for item in batch]
    seqs_padded = pad_sequence(seqs, batch_first=True, padding_value=PAD_TOKEN)
    
    ids = [item['id'] for item in batch]
    
    batch_out = {
        'seq': seqs_padded,
        'ids': ids,
        'lengths': torch.tensor([len(s) for s in seqs], dtype=torch.long)
    }
    
    if 'q8' in batch[0]:
        q8s = [item['q8'] for item in batch]
        q3s = [item['q3'] for item in batch]
        
        batch_out['q8'] = pad_sequence(q8s, batch_first=True, padding_value=PAD_TOKEN)
        batch_out['q3'] = pad_sequence(q3s, batch_first=True, padding_value=PAD_TOKEN)
        
    return batch_out

# Create Datasets
train_ds = ProteinDataset(cfg.train_path, mode='train')
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_fn)

print(f"Train samples: {len(train_ds)}")
batch = next(iter(train_loader))
print(f"Sample Batch Shapes: Seq={batch['seq'].shape}, Q8={batch['q8'].shape}")

Vocab Sizes: AA=23, Q8=9, Q3=4
Train samples: 7262
Sample Batch Shapes: Seq=torch.Size([64, 683]), Q8=torch.Size([64, 683])


## Model Architecture (Vanilla BiRNN)

In [5]:
class BiRNN_MTL(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, num_classes_q8, num_classes_q3):
        super(BiRNN_MTL, self).__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_TOKEN)
        
        # Vanilla RNN (bidirectional)
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )
        
        # Linear Heads
        # Input to linear is hidden_dim * 2 because of bidirectional
        self.q8_head = nn.Linear(hidden_dim * 2, num_classes_q8)
        self.q3_head = nn.Linear(hidden_dim * 2, num_classes_q3)
        
    def forward(self, x):
        # x: [batch, seq_len]
        embed = self.embedding(x)
        
        # RNN Output: [batch, seq_len, hidden_dim * 2]
        output, _ = self.rnn(embed)
        
        # Heads
        q8_logits = self.q8_head(output)
        q3_logits = self.q3_head(output)
        
        return q8_logits, q3_logits

model = BiRNN_MTL(
    vocab_size=len(AA_TO_ID),
    embedding_dim=cfg.embed_dim,
    hidden_dim=cfg.hidden_dim,
    num_layers=cfg.num_layers,
    num_classes_q8=len(Q8_TO_ID),
    num_classes_q3=len(Q3_TO_ID)
).to(cfg.device)

print(model)

BiRNN_MTL(
  (embedding): Embedding(23, 128, padding_idx=0)
  (rnn): RNN(128, 256, num_layers=2, batch_first=True, bidirectional=True)
  (q8_head): Linear(in_features=512, out_features=9, bias=True)
  (q3_head): Linear(in_features=512, out_features=4, bias=True)
)


## Training Routine

In [6]:
optimizer = optim.Adam(model.parameters(), lr=cfg.learning_rate)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN)

def calculate_metric(preds, targets, pad_idx=0):
    # Flatten
    preds_flat = preds.argmax(dim=-1).view(-1).cpu().numpy()
    targets_flat = targets.view(-1).cpu().numpy()
    
    # Filter padding
    mask = targets_flat != pad_idx
    preds_masked = preds_flat[mask]
    targets_masked = targets_flat[mask]
    
    return f1_score(targets_masked, preds_masked, average='macro')

def train_epoch(model, loader, optimizer, criterion, device, epoch):
    model.train()
    total_loss = 0
    total_q8_f1 = 0
    total_q3_f1 = 0
    count = 0
    
    pbar = tqdm(loader, desc=f"Epoch {epoch+1} Train")
    for batch in pbar:
        seq = batch['seq'].to(device)
        q8_tgt = batch['q8'].to(device)
        q3_tgt = batch['q3'].to(device)
        
        optimizer.zero_grad()
        
        q8_logits, q3_logits = model(seq)
        
        # Reshape for Loss: [Batch*Seq, Classes]
        loss_q8 = criterion(q8_logits.view(-1, len(Q8_TO_ID)), q8_tgt.view(-1))
        loss_q3 = criterion(q3_logits.view(-1, len(Q3_TO_ID)), q3_tgt.view(-1))
        
        loss = loss_q8 + loss_q3
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Metrics (approx batch-wise)
        f1_q8 = calculate_metric(q8_logits.detach(), q8_tgt, PAD_TOKEN)
        f1_q3 = calculate_metric(q3_logits.detach(), q3_tgt, PAD_TOKEN)
        total_q8_f1 += f1_q8
        total_q3_f1 += f1_q3
        count += 1
        
        pbar.set_postfix({'loss': loss.item(), 'f1_q8': f1_q8, 'f1_q3': f1_q3})
        
        if trk:
            trk.log({
                'train/loss': loss.item(),
                'train/f1_q8': f1_q8,
                'train/f1_q3': f1_q3,
                'train/loss_q8': loss_q8.item(),
                'train/loss_q3': loss_q3.item()
            })
            
    return total_loss/count, total_q8_f1/count, total_q3_f1/count

# Main Loop
for epoch in range(cfg.epochs):
    loss, f1_8, f1_3 = train_epoch(model, train_loader, optimizer, criterion, cfg.device, epoch)
    print(f"Epoch {epoch+1} Summary: Loss={loss:.4f}, F1_Q8={f1_8:.4f}, F1_Q3={f1_3:.4f}")
    # Saving model logic could go here

Epoch 1 Train:  16%|█▌        | 18/114 [00:28<03:57,  2.47s/it, loss=2.28, f1_q8=0.236, f1_q3=0.595]Exception in thread Thread-15 (_init_client_background):
Traceback (most recent call last):
  File "/Users/manyakaistha/.local/share/uv/python/cpython-3.10.18-macos-aarch64-none/lib/python3.10/threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "/Users/manyakaistha/.local/share/uv/python/cpython-3.10.18-macos-aarch64-none/lib/python3.10/threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "/Volumes/1700 APFS/PERSONAL_PROGRAMMING_PROJECTS/intro_dl_nppe/nppe2/.venv/lib/python3.10/site-packages/trackio/run.py", line 111, in _init_client_background
    self._batch_sender()
  File "/Volumes/1700 APFS/PERSONAL_PROGRAMMING_PROJECTS/intro_dl_nppe/nppe2/.venv/lib/python3.10/site-packages/trackio/run.py", line 82, in _batch_sender
    self._client.predict(
  File "/Volumes/1700 APFS/PERSONAL_PROGRAMMING_PROJECTS/intro_dl_nppe/nppe2/.venv/lib/python3.

KeyboardInterrupt: 

## Inference

In [ ]:
test_ds = ProteinDataset(cfg.test_path, mode='test')
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn)

model.eval()
results = []

print("Starting Inference...")
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Inference"):
        seq = batch['seq'].to(cfg.device)
        ids = batch['ids']
        lengths = batch['lengths']
        
        q8_logits, q3_logits = model(seq)
        
        q8_preds = q8_logits.argmax(dim=-1).cpu().numpy()
        q3_preds = q3_logits.argmax(dim=-1).cpu().numpy()
        
        for i, id_ in enumerate(ids):
            length = lengths[i].item()
            
            # Extract and truncate
            pred_q8_ids = q8_preds[i, :length]
            pred_q3_ids = q3_preds[i, :length]
            
            # Decode
            pred_q8_str = ''.join([ID_TO_Q8[idx] for idx in pred_q8_ids])
            pred_q3_str = ''.join([ID_TO_Q3[idx] for idx in pred_q3_ids])
            
            results.append({
                'id': id_,
                'sst8': pred_q8_str,
                'sst3': pred_q3_str
            })

df_submission = pd.DataFrame(results)
df_submission.to_csv('submission.csv', index=False)
print("Submission saved to submission.csv")
df_submission.head()